# Getting Started with EVōC

Welcome! This notebook will walk you through the fundamentals of EVōC, showing you how it works on both synthetic and real data. We'll start simple and progressively explore more advanced features.

The goal of this notebook is to give you hands-on experience with EVōC's key capabilities:
- Automatic cluster discovery without knowing k upfront
- Confidence scores (membership strengths) for each assignment
- Multiple clustering granularities at different resolutions
- How parameters affect the results

Let's dive in!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from evoc import EVoC

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

## Part 1: Understanding EVōC with Synthetic Data

Let's start with a controlled example: synthetic data where we know the true cluster structure. This lets us see exactly how well EVōC recovers the underlying clusters.

We'll create 1,000 points in 128 dimensions with 5 natural clusters. The high dimensionality (128D) mimics real embeddings from models like CLIP or sentence transformers, and EVōC is specifically optimized for this type of data.

In [ ]:
# Generate synthetic data: 1000 points in 128 dimensions with 5 natural clusters
X_synthetic, y_true = make_blobs(
    n_samples=1000,
    n_features=128,
    centers=5,
    random_state=42,
    cluster_std=1.5
)

# Normalize the data (standard practice for embeddings)
scaler = StandardScaler()
X_synthetic = scaler.fit_transform(X_synthetic).astype(np.float32)

print(f"Synthetic dataset shape: {X_synthetic.shape}")
print(f"True clusters in data: {len(np.unique(y_true))}")

### Running EVōC

Now we'll fit EVōC to our synthetic data. Notice that we don't need to specify how many clusters we expect—EVōC figures that out automatically. We just set `random_state` for reproducibility.

In [ ]:
# Create and fit the clusterer
clusterer = EVoC(random_state=42)
clusterer.fit(X_synthetic)

# Get results
labels = clusterer.labels_

print(f"\nClustering Results:")
print(f"  Number of clusters found: {len(np.unique(labels[labels >= 0]))}")
print(f"  Noise points: {np.sum(labels == -1)}")
print(f"  Cluster layer count: {len(clusterer.cluster_layers_)}")

### Understanding EVōC's Output

Now let's explore what EVōC gives us. Unlike simple clustering algorithms that return just one label per point, EVōC provides multiple pieces of information that give us confidence and insight into the results.

In [ ]:
# 1. Labels - which cluster each point belongs to
print("1. LABELS (cluster assignments):")
print(f"   Shape: {labels.shape}")
print(f"   First 20 labels: {labels[:20]}")
print(f"   Unique labels: {np.unique(labels)}")
print(f"   Note: -1 indicates noise points (not in any cluster)\n")

# 2. Membership strengths - confidence in each assignment
strengths = clusterer.membership_strengths_
print("2. MEMBERSHIP STRENGTHS (confidence scores):")
print(f"   Shape: {strengths.shape}")
print(f"   First 20 strengths: {strengths[:20]}")
print(f"   Range: [{strengths.min():.3f}, {strengths.max():.3f}]")
print(f"   Average: {strengths.mean():.3f}\n")

# 3. Cluster layers - multiple granularities
print("3. CLUSTER LAYERS (multiple granularities):")
print(f"   Number of layers: {len(clusterer.cluster_layers_)}")
for i, layer in enumerate(clusterer.cluster_layers_):
    n_clusters = len(np.unique(layer[layer >= 0]))
    n_noise = np.sum(layer == -1)
    persistence = clusterer.persistence_scores_[i]
    print(f"   Layer {i}: {n_clusters:3d} clusters, {n_noise:4d} noise, persistence={persistence:.3f}")

### Evaluating Cluster Quality

We created this data with 5 true clusters, so let's see if EVōC recovered them. Since we have ground truth labels, we can use standard clustering metrics to measure how well EVōC performed.

The Adjusted Rand Index (ARI) and Adjusted Mutual Information (AMI) both measure agreement between two clusterings, with 1.0 being perfect agreement. We'll only evaluate non-noise points for a fair comparison—noise points are supposed to be different from the true clusters.

In [ ]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# Get points that aren't noise (for fair comparison)
non_noise_mask = labels >= 0

# Compare with ground truth (only for non-noise points)
true_labels_subset = y_true[non_noise_mask]
pred_labels_subset = labels[non_noise_mask]

ari = adjusted_rand_score(true_labels_subset, pred_labels_subset)
ami = adjusted_mutual_info_score(true_labels_subset, pred_labels_subset)

print(f"Quality Metrics (comparing with ground truth):")
print(f"  Adjusted Rand Index (ARI): {ari:.3f}")
print(f"  Adjusted Mutual Information (AMI): {ami:.3f}")
print(f"\n  Note: Values close to 1.0 indicate perfect clustering")
print(f"  Note: These metrics only evaluate non-noise points ({non_noise_mask.sum()} out of {len(labels)})")

### Understanding Membership Strengths

One of EVōC's most useful features is the **membership strength** score for each point. This tells us how confident the algorithm is about each cluster assignment, ranging from 0 (very uncertain) to 1 (very confident).

Think of it this way: core cluster members have high strength because they're deep within the cluster. Boundary points have lower strength because they're near the edges or between clusters. Points marked as noise have strength 0 because they don't belong to any cluster.

Let's categorize our points based on confidence:

In [ ]:
# Separate by confidence level
core_mask = (labels >= 0) & (strengths > 0.8)
boundary_mask = (labels >= 0) & (strengths <= 0.8) & (strengths > 0.5)
noise_mask = labels == -1
uncertain_mask = (labels >= 0) & (strengths <= 0.5)

print("Point Classification by Confidence:")
print(f"  Core cluster points (strength > 0.8):    {core_mask.sum():5d}")
print(f"  Boundary points (0.5 < strength ≤ 0.8):  {boundary_mask.sum():5d}")
print(f"  Uncertain points (strength ≤ 0.5):       {uncertain_mask.sum():5d}")
print(f"  Noise points:                            {noise_mask.sum():5d}")
print(f"  Total:                                   {len(labels):5d}")

# Average strength per point type
if core_mask.sum() > 0:
    print(f"\nAverage strength by type:")
    print(f"  Core:      {strengths[core_mask].mean():.3f}")
    print(f"  Boundary:  {strengths[boundary_mask].mean():.3f}")
    if uncertain_mask.sum() > 0:
        print(f"  Uncertain: {strengths[uncertain_mask].mean():.3f}")

## Part 2: Exploring Hierarchical Structure

Here's something unique about EVōC: it doesn't just give you one clustering. It gives you a whole hierarchy of clusterings at different resolutions, all discovered in a single pass.

Think of it like zooming in and out on your data:
- At the **coarsest level** (higher layers), you see the big picture with few clusters
- At the **finest level** (lower layers), you see detailed structure with many clusters
- In **between**, you see intermediate granularities

This is invaluable when you want to explore your data at different scales. Let's see what's in the layer hierarchy:

In [ ]:
# Let's compare clustering quality across different layers
print("Layer-by-layer Analysis:\n")

results = []
for i, layer in enumerate(clusterer.cluster_layers_):
    n_clusters = len(np.unique(layer[layer >= 0]))
    n_noise = np.sum(layer == -1)
    
    # Quality on non-noise points
    layer_non_noise = layer >= 0
    if layer_non_noise.sum() > 0:
        true_subset = y_true[layer_non_noise]
        pred_subset = layer[layer_non_noise]
        ari = adjusted_rand_score(true_subset, pred_subset)
    else:
        ari = 0
    
    results.append({
        'layer': i,
        'clusters': n_clusters,
        'noise': n_noise,
        'ari': ari,
        'persistence': clusterer.persistence_scores_[i]
    })
    
    print(f"Layer {i}: {n_clusters:2d} clusters, {n_noise:3d} noise, ARI={ari:.3f}, persistence={clusterer.persistence_scores_[i]:.3f}")

# Find best layer
best_layer = max(results, key=lambda x: x['ari'])
print(f"\nBest layer (by ARI): Layer {best_layer['layer']} with ARI={best_layer['ari']:.3f}")
print(f"Default layer: 0 (automatically selected by EVōC)")

## Part 3: How Parameters Change Behavior

So far we've used default parameters, and EVōC worked well. But what if the default behavior doesn't match your needs? Let's explore how the main parameter, `noise_level`, affects clustering.

The `noise_level` parameter controls how aggressive EVōC is about identifying noise:
- **Lower values** (e.g., 0.2) → more clusters, fewer noise points (less filtering)
- **Higher values** (e.g., 0.8) → fewer clusters, more noise points (more filtering)
- **Default value** (0.5) → balanced approach

Let's see this in action:

In [ ]:
# Try different noise levels
noise_levels = [0.2, 0.4, 0.5, 0.6, 0.8]

print("Effect of noise_level parameter:\n")
print("noise_level | clusters | noise pts | quality (ARI)")
print("-" * 55)

for noise_level in noise_levels:
    clusterer_nl = EVoC(noise_level=noise_level, random_state=42)
    labels_nl = clusterer_nl.fit_predict(X_synthetic)
    
    n_clusters = len(np.unique(labels_nl[labels_nl >= 0]))
    n_noise = np.sum(labels_nl == -1)
    
    # Quality
    non_noise = labels_nl >= 0
    if non_noise.sum() > 0:
        ari = adjusted_rand_score(y_true[non_noise], labels_nl[non_noise])
    else:
        ari = 0
    
    print(f"    {noise_level:.1f}    |    {n_clusters:2d}    |    {n_noise:3d}    |    {ari:.3f}")

print("\nObservations:")
print("  - Lower noise_level → more clusters, fewer noise points")
print("  - Higher noise_level → fewer clusters, more noise points")
print("  - Default (0.5) balances discovery with noise filtering")

## Part 4: Real Data - Clustering MNIST Digits

Now let's move from synthetic data to something real: the MNIST handwritten digits dataset. This has 70,000 images of digits (0-9), which gives us true labels to evaluate against.

We'll use a 5,000-sample subset for speed, but EVōC scales efficiently to much larger datasets. Each image is represented as 784 pixel values (28×28 pixels), which we'll normalize to [0,1].

In [ ]:
# Load MNIST
from sklearn.datasets import fetch_openml

print("Loading MNIST... (this may take a minute)")
mnist = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
X_mnist, y_mnist = mnist

# Use subset for speed (first 5000 samples)
X_mnist = X_mnist[:5000].astype(np.float32) / 255.0
y_mnist = y_mnist[:5000].astype(int)

print(f"\nMNIST dataset (subset):")
print(f"  Shape: {X_mnist.shape}")
print(f"  True classes: {len(np.unique(y_mnist))} (digits 0-9)")

In [ ]:
# Cluster MNIST
print("Clustering MNIST with EVōC...")
clusterer_mnist = EVoC(n_neighbors=15, random_state=42)
labels_mnist = clusterer_mnist.fit_predict(X_mnist)

print(f"\nResults:")
print(f"  Clusters found: {len(np.unique(labels_mnist[labels_mnist >= 0]))}")
print(f"  Noise points: {np.sum(labels_mnist == -1)}")
print(f"  Available layers: {len(clusterer_mnist.cluster_layers_)}")

# Quality vs ground truth
non_noise = labels_mnist >= 0
if non_noise.sum() > 0:
    ari = adjusted_rand_score(y_mnist[non_noise], labels_mnist[non_noise])
    ami = adjusted_mutual_info_score(y_mnist[non_noise], labels_mnist[non_noise])
    print(f"\nQuality (vs 10 true digit classes):")
    print(f"  ARI: {ari:.3f}")
    print(f"  AMI: {ami:.3f}")

print(f"\nInterpretation:")
print(f"  Our 10 digits became {len(np.unique(labels_mnist[labels_mnist >= 0]))} clusters.")
print(f"  This is natural: some digits visually merge (like 0 and 8), while others split.")


### Exploring MNIST at Different Granularities

Just like with synthetic data, EVōC discovered a hierarchy of clusterings. The MNIST case is interesting because we have 10 true digit classes, but EVōC's clustering might group them differently at different layers.

Let's examine the first several layers to see how the granularity changes:

In [ ]:
# Explore MNIST layers
print("MNIST clustering at different granularities:\n")
print("Layer | Clusters | Noise | Persistence | ARI (vs digit labels)")
print("-" * 65)

for i, layer in enumerate(clusterer_mnist.cluster_layers_[:5]):  # Show first 5
    n_clusters = len(np.unique(layer[layer >= 0]))
    n_noise = np.sum(layer == -1)
    persistence = clusterer_mnist.persistence_scores_[i]
    
    layer_non_noise = layer >= 0
    if layer_non_noise.sum() > 0:
        ari = adjusted_rand_score(y_mnist[layer_non_noise], layer[layer_non_noise])
    else:
        ari = 0
    
    print(f"  {i}   |    {n_clusters:2d}    | {n_noise:4d} |    {persistence:.3f}    |    {ari:.3f}")

print(f"\nNote: Digit 0 and 1 might merge into one cluster (visually similar),")
print(f"while others split into multiple clusters. This is natural behavior.")

## Summary & Key Insights

We've covered a lot in this notebook! Let's recap what we learned:

### What We Discovered

1. **Automatic Cluster Discovery** — EVōC found 5 clusters in synthetic data without being told k upfront. With MNIST, it discovered structure based on visual similarity, not just ground truth labels.

2. **Confidence Scores** — Membership strengths let us separate confident assignments from uncertain boundary cases. This is crucial when you need to trust your clusters.

3. **Multiple Granularities** — The layer hierarchy gives us flexibility. Different layers work better for different tasks:
   - Fine layers: detailed structure, good for exploration
   - Coarse layers: big picture, good for summarization
   - Intermediate layers: practical balance

4. **Parameter Effects** — The `noise_level` parameter provides a simple knob to control discovery vs. noise filtering. Default (0.5) works well for most cases.

5. **Real-World Performance** — EVōC recovered sensible structure from actual digit images, achieving good agreement with ground truth labels despite the inherent difficulty of the task.

### When to Use Which

- **Use the default clustering (layer 0)** for most purposes
- **Use membership strengths** when you need confidence estimates
- **Explore different layers** when you want multiple perspectives on your data
- **Tune `noise_level`** only if defaults don't match your specific needs

### Next Steps

Now you understand the basics! Here's what you might explore next:

1. **Try your own data** — Load embeddings from your models and see what structures EVōC discovers
2. **Compare with alternatives** — See the `06_performance_benchmarks` notebook for comparisons with K-Means and UMAP+HDBSCAN
3. **Understand the algorithm** — Read the `how_evoc_works` guide for intuition about what's happening under the hood
4. **Explore advanced features** — The `07_understanding_layers` notebook deep-dives into layer selection strategies and use cases
5. **See domain examples** — Text embeddings, image embeddings, and more in dedicated notebooks

Happy clustering!